# 11b — Estimação causal: T2/T3 dose-response × 3 snapshots × 5 canais AFOLU × 4 specs PSM

**Escopo da Fase B3.3 (§11.2 v2.3.5):**
- **Tratamentos:** T2 (conformidade NEEA, %) e T3 (intensidade gCO₂eq/MJ), cada um em 3 snapshots ANP (2022, 2025, 2026).
- **Outcomes:** 5 canais AFOLU transformados.
- **Specs PSM:** LEAN(21) / FULL(35) / FULL−2(33) / RICH(52).
- **Estimador:** CS multi-valor (Callaway, Goodman-Bacon & Sant'Anna 2024) via `differences.ATTgt(dosage_column=...)` com `est_method='dr'`.
- **Total ATTs:** 2 (T2/T3) × 3 (snapshots) × 5 (outcomes) × 4 (specs) = **120 ATTs**.
- **Bootstrap:** n_boot=199 (desenvolvimento). Aumentar para 999 antes da submissão final.

**Decisão metodológica (v2.3.5):** dose discretizada em **5 níveis** (0 = never-treated OU tratado sem dose no snapshot; 1-4 = quartis da dose entre tratados com dose>0). Motivação: CS multi-valor com dose contínua de alta cardinalidade (110+ valores únicos) é computacionalmente instável em `differences`. Discretização em quartis preserva interpretação dose-response e respeita estrutura empírica do CGS-2024.\n\n**Política de cobertura:** os 194 tratados ficam todos na amostra. Tratados sem dose observada em um snapshot (canceladas, suspensas, ou certificadas após o snapshot) recebem `dose = 0`, interpretado substantivamente como 'tratado sem conformidade efetiva naquele snapshot'. Cobertura efetiva: ~60% de tratados com dose > 0 em cada snapshot. **Caveat substantivo:** a interpretação dose-response assume tratamento absorvente — descredenciamentos pós-tratamento configuram caso limite que deve ser caveado no paper (§3.5.3 v2.3.5).

**Inputs:**
- `data/interim/panel_canavieiro_main.csv` (842 munis × 10 anos)
- `data/raw/psm_baseline/base_psm_integrada_raw.csv` (165 cols)

**Outputs em `data/interim/`:**
- `att_t2t3_main.csv` — Tabela 3 do paper (120 ATTs)
- `att_t2t3_eventstudy_luc.csv` — event-study para LUC sob snapshot principal

**Tempo esperado:** 15-25 min com n_boot=199; ~60-90 min com n_boot=999.

## Setup

In [ ]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
!pip install -q differences

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from differences import ATTgt

from pipeline.config import interim, out_pre

# Bootstrap config — ALTERAR PARA 999 ANTES DA SUBMISSÃO FINAL
N_BOOT = 199
RANDOM_STATE = 42

print(f'✓ setup OK (bootstrap n={N_BOOT}, seed={RANDOM_STATE})')

## Bloco 1 — Carregar painel e preparar covariáveis

In [ ]:
panel = pd.read_csv(interim('panel_canavieiro_main.csv'), dtype={'geocode': str})

# Dropar covs colidentes (mesma estratégia do 11a)
COVS_COLIDENTES = ['gini', 'densidade_pop', 'log_pop', 'idhm_renda', 
                    'ivs_capital_humano', 'ivs_renda_trabalho']
panel = panel.drop(columns=[c for c in COVS_COLIDENTES if c in panel.columns])
print(f'panel após dropar colidentes: {panel.shape}')

# Verificar colunas dose T2/T3
dose_cols = [c for c in panel.columns if c.startswith('dose_T2_') or c.startswith('dose_T3_')]
print(f'\nColunas dose disponíveis: {dose_cols}')

# Diagnóstico de cobertura por snapshot
print(f'\nCobertura de dose entre tratados (194 munis):')
muni = panel.groupby('geocode').first().reset_index()
treated = muni[muni['g_m'].notna()]
for col in dose_cols:
    nn = treated[col].notna().sum()
    print(f'  {col:25s}: {nn}/194 ({100*nn/194:.1f}%)')

In [ ]:
# Reconstruir covariáveis derivadas (mesma função do notebook 10/11a)
psm_raw = pd.read_csv(
    BASE_DIR / 'data/raw/psm_baseline/base_psm_integrada_raw.csv',
    low_memory=False,
)
psm_raw['geocode'] = psm_raw['0_cd_ibge'].astype(str).str.zfill(7)

BIOMA_FIXES = {'Amaz\ufffd\ufffdnia': 'Amazônia', 'Mata Atl\ufffd\ufffdntica': 'Mata Atlântica'}
if '14_bioma' in psm_raw.columns:
    psm_raw['14_bioma'] = psm_raw['14_bioma'].replace(BIOMA_FIXES)

muni_id = (panel.groupby('geocode', as_index=False)
           .agg(municipio=('municipio','first'), uf=('uf','first'),
                is_treated_ever=('is_treated_ever','first'), g_m=('g_m','first'),
                bioma=('bioma','first')))
muni_id['treated'] = muni_id['is_treated_ever'].astype(int)

df_cs = muni_id.merge(psm_raw, on='geocode', how='inner')
print(f'df_cs (cross-section): {df_cs.shape}')

In [ ]:
# Helpers e build_covariates_raw
def safe_log1p(s, idx):
    s = pd.to_numeric(s, errors='coerce') if s is not None else pd.Series(np.nan, index=idx)
    return np.log1p(s.clip(lower=0))
def safe_div(num, den, idx):
    num = pd.to_numeric(num, errors='coerce') if num is not None else pd.Series(np.nan, index=idx)
    den = pd.to_numeric(den, errors='coerce') if den is not None else pd.Series(np.nan, index=idx)
    return np.where((den.notna()) & (den > 0), num / den, np.nan)
def asn(s, idx):
    return pd.to_numeric(s, errors='coerce') if s is not None else pd.Series(np.nan, index=idx)

def build_covariates_raw(df):
    d = df.copy(); idx = d.index
    d['log_pib_total']=safe_log1p(d.get('1_pib_total'),idx)
    d['log_pib_pc']=safe_log1p(d.get('1_pib_percap'),idx)
    d['log_pop']=safe_log1p(d.get('2_pop_2017_ibge'),idx)
    d['log_area_total']=safe_log1p(d.get('14_area_total'),idx)
    d['densidade_pop']=safe_div(d.get('2_pop_2017_ibge'),d.get('14_area_total'),idx)
    d['share_vadc_agro']=safe_div(d.get('1_vadc_agro'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_ind']=safe_div(d.get('1_vadc_ind'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_serv']=safe_div(d.get('1_vadc_serv'),d.get('1_vadc_bruto'),idx)
    d['share_vadc_adm']=safe_div(d.get('1_vadc_adm'),d.get('1_vadc_bruto'),idx)
    d['share_cana_baseline']=asn(d.get('3_mb_sharegrp_pre_cana'),idx)
    d['mb_share_soja']=asn(d.get('3_mb_sharegrp_pre_soja'),idx)
    d['mb_share_pastagem']=asn(d.get('3_mb_sharegrp_pre_pastagem'),idx)
    d['mb_share_vegetacao_nativa']=asn(d.get('3_mb_sharegrp_pre_vegetacao_nativa'),idx)
    d['mb_share_urbano']=asn(d.get('3_mb_sharegrp_pre_urbano_infra'),idx)
    d['log_area_cana']=safe_log1p(d.get('4_area_colhida_ha_cana'),idx)
    d['log_area_soja']=safe_log1p(d.get('4_area_colhida_ha_soja'),idx)
    d['log_area_agri_total']=safe_log1p(d.get('4_area_colhida_ha'),idx)
    d['share_area_cana_agri']=safe_div(d.get('4_area_colhida_ha_cana'),d.get('4_area_colhida_ha'),idx)
    d['share_est_af']=safe_div(d.get('5_num_est_af'),d.get('5_num_est_total'),idx)
    d['share_est_mp']=safe_div(d.get('5_num_est_mp'),d.get('5_num_est_total'),idx)
    d['share_area_af']=safe_div(d.get('6_area_lav_af'),d.get('6_area_lav_total'),idx)
    d['share_area_mp']=safe_div(d.get('6_area_lav_mp'),d.get('6_area_lav_total'),idx)
    d['trator_per_est']=safe_div(d.get('11_num_trator_total'),d.get('5_num_est_total'),idx)
    d['share_est_irrig']=safe_div(d.get('12_num_est_irrig_total'),d.get('5_num_est_total'),idx)
    d['share_area_irrig']=safe_div(d.get('12_area_irrig_total'),d.get('6_area_lav_total'),idx)
    d['share_est_fin_total']=safe_div(d.get('13_num_est_fin_total'),d.get('5_num_est_total'),idx)
    d['share_est_at']=safe_div(d.get('10_num_est_receb_at'),d.get('5_num_est_total'),idx)
    d['natveg_share_area']=safe_div(d.get('14_vegetacao_natural'),d.get('14_area_total'),idx)
    d['desmat_share_area']=safe_div(d.get('14_desmatado'),d.get('14_area_total'),idx)
    d['idhm_renda']=asn(d.get('17_idhm_renda'),idx); d['idhm_educ']=asn(d.get('17_idhm_educ'),idx)
    d['ivs_infra']=asn(d.get('17_ivs_infraestrutura_urbana'),idx); d['gini']=asn(d.get('17_i_gini'),idx)
    d['idhm_long']=asn(d.get('17_idhm_long'),idx); d['ivs_capital_humano']=asn(d.get('17_ivs_capital_humano'),idx)
    d['ivs_renda_trabalho']=asn(d.get('17_ivs_renda_e_trabalho'),idx)
    d['share_est_at_coop']=safe_div(d.get('10_num_est_receb_at_coop'),d.get('5_num_est_total'),idx)
    d['share_est_at_gov']=safe_div(d.get('10_num_est_receb_at_gov'),d.get('5_num_est_total'),idx)
    d['share_fin_invest']=safe_div(d.get('13_num_est_fin_invest'),d.get('5_num_est_total'),idx)
    d['share_fin_cust']=safe_div(d.get('13_num_est_fin_cust'),d.get('5_num_est_total'),idx)
    d['share_est_trator']=safe_div(d.get('11_num_est_trator_total'),d.get('5_num_est_total'),idx)
    d['share_est_irrig_pivo']=safe_div(d.get('12_num_est_irrig_pivo'),d.get('5_num_est_total'),idx)
    d['pct_est_energia']=asn(d.get('7_est_com_energia%'),idx)
    d['log_area_milho']=safe_log1p(d.get('4_area_colhida_ha_milho'),idx)
    d['log_area_alg']=safe_log1p(d.get('4_area_colhida_ha_alg'),idx)
    d['log_area_cafarab']=safe_log1p(d.get('4_area_colhida_ha_cafarab'),idx)
    d['mb_share_urbano']=asn(d.get('3_mb_sharegrp_pre_urbano_infra'),idx)
    d['mb_share_agua']=asn(d.get('3_mb_sharegrp_pre_agua'),idx)
    d['mb_share_outros']=asn(d.get('3_mb_sharegrp_pre_outros'),idx)
    d['mb_share_agri_total']=asn(d.get('3_mb_sharegrp_pre_agricultura_total'),idx)
    d['share_num_est_mp']=safe_div(d.get('5_num_est_mp'),d.get('5_num_est_total'),idx)
    d['share_est_pec']=safe_div(d.get('5_num_est_pec_total'),d.get('5_num_est_total'),idx)
    d['share_est_lavperm']=safe_div(d.get('5_num_est_lavperm_total'),d.get('5_num_est_total'),idx)
    d['share_est_lavtemp']=safe_div(d.get('5_num_est_lavtemp_total'),d.get('5_num_est_total'),idx)
    d['share_area_lavperm']=safe_div(d.get('6_area_lavperm_total'),d.get('6_area_lav_total'),idx)
    d['share_area_lavtemp']=safe_div(d.get('6_area_lavtemp_total'),d.get('6_area_lav_total'),idx)
    d['share_area_pec']=safe_div(d.get('6_area_pec_total'),d.get('6_area_lav_total'),idx)
    d['share_fin_comer']=safe_div(d.get('13_num_est_fin_comer'),d.get('5_num_est_total'),idx)
    d['share_est_at_propr']=safe_div(d.get('10_num_est_receb_at_propr'),d.get('5_num_est_total'),idx)
    d['share_est_at_gov_out']=safe_div(d.get('10_num_est_receb_at_gov_out'),d.get('5_num_est_total'),idx)
    share_cols = [c for c in d.columns if ('share' in c) or c.startswith('mb_share_')]
    for c in share_cols:
        s = pd.to_numeric(d[c], errors='coerce')
        if not s.dropna().empty and (s.dropna().between(-0.05,1.05).mean() > 0.8):
            d[c] = s.clip(0,1)
    return d

df_cs = build_covariates_raw(df_cs)
print(f'✓ Covariáveis reconstruídas')

In [ ]:
# Specs PSM (RICH-52, alinhado com v2.3.5)
COVS_LEAN = ['log_pib_total','log_pib_pc','log_pop','densidade_pop',
             'share_vadc_agro','share_vadc_ind','share_vadc_serv',
             'share_cana_baseline','mb_share_soja','mb_share_pastagem','mb_share_vegetacao_nativa',
             'log_area_cana','log_area_soja','share_area_cana_agri',
             'share_est_af','share_area_af','trator_per_est','share_est_irrig','share_est_fin_total',
             'ivs_infra','gini']
COVS_FULL = COVS_LEAN + ['share_vadc_adm','idhm_educ','idhm_renda','idhm_long',
                          'ivs_capital_humano','ivs_renda_trabalho',
                          'share_est_at','share_est_at_coop','share_est_at_gov',
                          'share_fin_invest','share_fin_cust',
                          'share_est_trator','share_est_irrig_pivo','pct_est_energia']
COVS_FULL2 = [c for c in COVS_FULL if c not in ('share_vadc_agro','share_vadc_ind')]
# RICH-52 (sem 4 covas quase-degeneradas, decisão v2.3.5)
COVS_RICH_FILTRADAS_OUT = ['mb_share_algodao','log_area_cafcan','mb_share_cafe','mb_share_silvicultura']
COVS_RICH = COVS_FULL + [c for c in [
    'log_area_milho','log_area_alg','log_area_cafarab','mb_share_urbano','mb_share_agua',
    'mb_share_outros','mb_share_agri_total','share_num_est_mp','share_est_pec',
    'share_est_lavperm','share_est_lavtemp','share_area_lavperm','share_area_lavtemp',
    'share_area_pec','share_fin_comer','share_est_at_propr','share_est_at_gov_out',
] if c not in COVS_RICH_FILTRADAS_OUT]
assert len(COVS_RICH) == 52, f'RICH tem {len(COVS_RICH)}, esperado 52'

SPECS = {'LEAN': COVS_LEAN, 'FULL': COVS_FULL, 'FULL2': COVS_FULL2, 'RICH': COVS_RICH}
for n, c in SPECS.items():
    print(f'  {n:6s}: {len(c)} covs')

# Imputação
all_covs = sorted(set(COVS_RICH))
for c in all_covs:
    if df_cs[c].isna().any():
        df_cs[c] = df_cs.groupby('uf')[c].transform(lambda x: x.fillna(x.median()))
        df_cs[c] = df_cs[c].fillna(df_cs[c].median())
assert df_cs[all_covs].isna().sum().sum() == 0
print(f'\n✓ Imputação OK')

## Bloco 2 — Preparar painel para CS multi-valor

Para CS multi-valor:
- `cohort_column`: `g_m_cs` (NaN para never-treated)
- `dosage_column`: uma coluna por snapshot × tratamento, com:
    - `dose > 0` para tratados com dose observada
    - `dose = 0` para tratados sem dose ou never-treated

**Política de cobertura:** os 194 tratados ficam todos. Tratados com `dose = 0` representam casos limite (canceladas, suspensas, ou certificadas após snapshot).

In [ ]:
# Merge covs ao painel longo
panel_cs = panel.merge(df_cs[['geocode'] + all_covs], on='geocode', how='left')
panel_cs['g_m_cs'] = panel_cs['g_m']  # NaN para never-treated

# Discretizar dose em quartis (decisão metodológica v2.3.5)
# CS multi-valor com dose contínua (110+ valores únicos) é instável computacionalmente em `differences`.
# Discretização em quintis (0=never/no-dose + 4 quartis para dose>0) preserva interpretação dose-response
# e respeita estrutura empírica do CGS-2024 que requer poucos níveis para inferência confiável.
TREATMENTS = ['T2', 'T3']
SNAPSHOTS = ['2022', '2025', '2026']
for t in TREATMENTS:
    for s in SNAPSHOTS:
        col_orig = f'dose_{t}_{s}'
        col_disc = f'{col_orig}_disc'
        dose_raw = panel_cs[col_orig].fillna(0)
        mask_pos = dose_raw > 0
        if mask_pos.sum() > 0:
            quartiles = pd.qcut(dose_raw[mask_pos], 4, labels=False, duplicates='drop')
            dose_disc = dose_raw.copy()
            dose_disc[mask_pos] = quartiles + 1  # quartis 1..4
            dose_disc[~mask_pos] = 0
            panel_cs[col_disc] = dose_disc.values
        else:
            panel_cs[col_disc] = 0

# Diagnóstico
print('Distribuição dos quartis de dose (0 = never-treated OR tratado sem dose):')
for t in TREATMENTS:
    for s in SNAPSHOTS:
        col = f'dose_{t}_{s}_disc'
        counts = panel_cs[col].value_counts().sort_index()
        print(f'  {col:30s}: {dict(counts)}')

panel_cs = panel_cs.set_index(['geocode', 'ano']).sort_index()
print(f'\n✓ Painel indexado: {panel_cs.shape}')

## Bloco 3 — Estimação CS multi-valor: 120 ATTs

Loop sobre 2 tratamentos × 3 snapshots × 5 outcomes × 4 specs.

**Agregação:** `aggregate('simple')` retorna ATT médio ponderado por cohort size. Para tratamento multi-valor, retorna ATT por *stratum* (valor de dose); a interpretação dose-response vem do plot por strata. Para a Tabela 3 do paper, reportamos:
1. **ATT médio agregado** sobre todos os strata de dose (= average treatment effect on the treated, ponderado)
2. **Slope dose-response** estimado por regressão linear sobre ATTs por strata

In [ ]:
import time

OUTCOMES = ['asinh_luc','asinh_carbono_solo','log1p_queima','log_solos_manejados','log1p_residuos_florestais']

results = []
t_total = time.time()

for treatment in TREATMENTS:
    for snapshot in SNAPSHOTS:
        dose_col = f'dose_{treatment}_{snapshot}_disc'
        print(f'\n{"="*70}\n>>> {treatment} (snapshot {snapshot})')
        
        for outcome in OUTCOMES:
            for spec_name, covs in SPECS.items():
                t_run = time.time()
                formula = f'{outcome} ~ ' + ' + '.join(covs)
                
                try:
                    data = panel_cs.dropna(subset=[outcome]).copy()
                    
                    attgt = ATTgt(
                        data=data,
                        cohort_column='g_m_cs',
                        dosage_column=dose_col,
                    )
                    attgt.fit(
                        formula=formula,
                        est_method='dr',
                        control_group='never_treated',
                        boot_iterations=N_BOOT,
                        random_state=RANDOM_STATE,
                        progress_bar=False,
                        n_jobs=1,
                    )
                    
                    # Agregação 'simple': ATT por stratum (= valor de dose). Tomamos média ponderada.
                    agg = attgt.aggregate('simple')
                    # `agg` é DataFrame indexed por strata (valores de dose)
                    # Pegamos média simples sobre strata para ATT global
                    att_arr = agg.iloc[:, 0].values
                    se_arr = agg.iloc[:, 1].values
                    
                    att_mean = np.nanmean(att_arr)
                    se_mean = np.sqrt(np.nanmean(se_arr ** 2))
                    ci_lo, ci_hi = att_mean - 1.96 * se_mean, att_mean + 1.96 * se_mean
                    
                    # Slope dose-response: regressão de ATT em valor de dose
                    doses = agg.index.values
                    if len(doses) >= 3:  # pelo menos 3 pontos
                        slope = np.polyfit(doses, att_arr, 1)[0]
                    else:
                        slope = np.nan
                    
                    results.append({
                        'treatment': treatment,
                        'snapshot': snapshot,
                        'outcome': outcome,
                        'spec': spec_name,
                        'ATT_mean': att_mean,
                        'SE_mean': se_mean,
                        'CI_lo': ci_lo,
                        'CI_hi': ci_hi,
                        'dose_slope': slope,
                        'n_strata': len(att_arr),
                        'n_munis': data.index.get_level_values('geocode').nunique(),
                        'time_s': time.time() - t_run,
                    })
                    print(f'  {outcome:30s} {spec_name:6s}  ATT={att_mean:+.4f}  slope={slope:+.5f}  [{time.time()-t_run:.1f}s]')
                except Exception as e:
                    print(f'  {outcome:30s} {spec_name:6s}  FALHOU: {type(e).__name__}: {str(e)[:60]}')
                    results.append({
                        'treatment': treatment, 'snapshot': snapshot,
                        'outcome': outcome, 'spec': spec_name,
                        'ATT_mean': np.nan, 'SE_mean': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan,
                        'dose_slope': np.nan, 'n_strata': 0, 'n_munis': 0, 'time_s': 0,
                    })

results_df = pd.DataFrame(results)
print(f'\n{"="*70}')
print(f'TOTAL: {results_df["ATT_mean"].notna().sum()}/{len(results_df)} sucessos em {time.time()-t_total:.1f}s')

## Bloco 4 — Salvar e mostrar Tabela 3 do paper

In [ ]:
# Salvar
results_df.to_csv(interim('att_t2t3_main.csv'), index=False)
print(f'✓ att_t2t3_main.csv salvo ({results_df.shape})')

# Resumo Tabela 3: para cada (treatment, snapshot, outcome), reportar ATT médio sob FULL (principal)
print('\n' + '='*80)
print('TABELA 3 — ATT médio sob FULL (principal) + slope dose-response')
print('='*80)

for treatment in TREATMENTS:
    print(f'\n--- {treatment} ---')
    for snapshot in SNAPSHOTS:
        print(f'\n  Snapshot {snapshot}:')
        sub = results_df.query('treatment == @treatment and snapshot == @snapshot and spec == "FULL"')
        for _, row in sub.iterrows():
            if pd.notna(row['ATT_mean']) and pd.notna(row['SE_mean']) and row['SE_mean'] > 0:
                t_stat = abs(row['ATT_mean'] / row['SE_mean'])
                star = '***' if t_stat > 2.58 else ('**' if t_stat > 1.96 else ('*' if t_stat > 1.65 else ''))
                print(f'    {row["outcome"]:30s} ATT={row["ATT_mean"]:+.4f} (SE={row["SE_mean"]:.4f}) {star}  slope={row["dose_slope"]:+.5f}')
            else:
                print(f'    {row["outcome"]:30s} FALHOU')

## Bloco 5 — Event-study LUC sob T2 snapshot 2026 (canal e snapshot principais)

In [ ]:
# Event-study sob FULL para LUC com T2 snapshot 2026 (principal)
outcome = 'asinh_luc'
dose_col = 'dose_T2_2026_disc'
covs = COVS_FULL

data_es = panel_cs.dropna(subset=[outcome]).copy()
attgt_es = ATTgt(data=data_es, cohort_column='g_m_cs', dosage_column=dose_col)
attgt_es.fit(
    formula=f'{outcome} ~ ' + ' + '.join(covs),
    est_method='dr',
    control_group='never_treated',
    boot_iterations=N_BOOT,
    random_state=RANDOM_STATE,
    progress_bar=False,
    n_jobs=1,
)

agg_event = attgt_es.aggregate('event')
print('Event-study CS multi-valor para LUC (T2, snapshot 2026):')
print(agg_event.head(15))

agg_event.to_csv(interim('att_t2t3_eventstudy_luc.csv'))
print('\n✓ Event-study salvo')

## Resumo final

Se rodou com sucesso:

1. **120 ATTs** salvos em `att_t2t3_main.csv` (2 tratamentos × 3 snapshots × 5 outcomes × 4 specs)
2. **Event-study** para LUC sob T2 snapshot 2026 (canal e snapshot principal)
3. Cada linha tem **ATT médio agregado sobre strata de dose** + **slope dose-response**

**Notas técnicas:**
- Política de cobertura: tratados sem dose recebem 0 (interpretado como 'sem conformidade efetiva no snapshot')
- CS multi-valor de Callaway-Goodman-Bacon-Sant'Anna 2024 ainda é estimador novo na literatura; a interpretação do `slope dose-response` requer cuidado (não é coeficiente causal de mediação, é elasticidade média condicional)
- Sensibilidade entre os 3 snapshots informa sobre estabilidade temporal do efeito (snapshots 2025/2026 incluem choque Lei 14.993/2024)

**TODO antes de submissão:**
- Aumentar `N_BOOT` para 999
- Plotar dose-response curves por outcome × snapshot (parte do paper)
- Reportar caveat sobre interpretação multi-valor no Apêndice